In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CanineModel, CanineTokenizer
import pandas as pd

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1225 10:40:06.374000 22976 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
class StyleEncoder(nn.Module):
    def __init__(self, model_name="google/canine-s", proj_dim=128):
        super().__init__()
        self.encoder = CanineModel.from_pretrained(model_name)
        hidden_dim = self.encoder.config.hidden_size

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Linear(256, proj_dim)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0]  # [CLS]
        z = self.proj(cls)
        z = F.normalize(z, dim=1)
        return z


In [3]:
def supervised_contrastive_loss(z, labels, temperature=0.1):
    """
    z: [B, D] normalized embeddings
    labels: [B] author ids
    """
    device = z.device
    B = z.size(0)

    sim = torch.matmul(z, z.T) / temperature
    sim = sim - torch.eye(B, device=device) * 1e9  # mask self

    labels = labels.unsqueeze(0)
    mask_pos = labels == labels.T

    exp_sim = torch.exp(sim)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True))

    mean_log_prob_pos = (mask_pos * log_prob).sum(dim=1) / mask_pos.sum(dim=1)

    loss = -mean_log_prob_pos.mean()
    return loss


In [4]:
# Example record
# {
#   "Author": "A",
#   "Content": "lol nahhh 😂😂 that was wild!!!"
# }

class ConversationDataset(torch.utils.data.Dataset):
    def __init__(self, rows, tokenizer, max_length=512):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

        # map authors to integer labels
        authors = sorted(set(r["Author"] for r in rows))
        self.author2id = {a: i for i, a in enumerate(authors)}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = self.tokenizer(
            row["Content"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "author": self.author2id[row["Author"]]
        }


In [5]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "author": torch.tensor([b["author"] for b in batch])
    }


In [7]:
rows = pd.read_csv("../data/raw/discord_messages_processed.csv").to_dict(orient="records")

tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
dataset = ConversationDataset(rows, tokenizer)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True
)
